## Install depedencies

In [ ]:
# %%capture --no-display
# !pip install newsapi-python
# !pip install -U langchain-community
# !pip install rouge_score
# !pip install tiktoken
# !pip install faiss-cpu
# !pip install sentence-transformers
# !pip install bertopic
# !pip install fuzzywuzzy
# !pip install pydub
# !pip install librosa
# !pip install rank_bm25 nltk

In [ ]:
# pip install openai==0.28

In [ ]:
# pip install datasets

In [ ]:
# pip install accelerate>=0.26.0

In [19]:
openai.__version__

'0.28.0'

In [1]:
import re
from sentence_transformers import SentenceTransformer, util
import statistics
import os
import requests
import pandas as pd
from datetime import datetime, timedelta

# from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import openai
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

In [ ]:
import os
import openai

# Set your API key (or use environment variable OPENAI_API_KEY)
openai.api_key = os.getenv("OPENAI_API_KEY", "your-api-key")

In [3]:
# Helper function to display markdown
from IPython.display import Markdown, display
def md(text):
    display(Markdown(text))

## Web Scraping

In [24]:
import requests
from bs4 import BeautifulSoup
import os
import re
import json

def fetch_and_save_transcript(url):
    # Send a GET request to the URL
    response = requests.get(url)
    response.raise_for_status()

    # Parse the HTML content
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find the article-body div
    article_body = soup.find('div', class_='article-body')
    if not article_body:
        raise ValueError("Could not find the article body.")

    # Remove all h2 tags from the article body
    for h2 in article_body.find_all('h2'):
        h2.decompose()

    # Extract all paragraph text
    paragraphs = article_body.find_all('p')
    transcript = '\n'.join([para.get_text() for para in paragraphs])

    # # Save to a .txt file
    # with open(filename, 'w', encoding='utf-8') as file:
    #     file.write(transcript)

    # print(f"Transcript saved to '{filename}'")
    return transcript


def chunk_transcript(text):
    # Pattern to detect speakers including Operator
    speaker_pattern = re.compile(r'^(?:[A-Z][a-z]+(?: [A-Z][a-z]+)* -- .+|Operator)$', re.MULTILINE)
    chunks = []

    matches = list(speaker_pattern.finditer(text))
    pending_question = None
    question_speaker = None
    chunk_id = 1

    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        speaker_line = match.group(0).strip()
        speaker_text = text[start:end].strip().replace('\n', ' ')

        # Skip empty text
        if not speaker_text:
            continue

        # Skip Operator chunks
        if speaker_line == "Operator":
            continue

        # Store analyst question temporarily
        if "Analyst" in speaker_line:
            pending_question = speaker_text
            question_speaker = speaker_line
            continue

        # Build final chunk with or without question
        chunk = {
            "chunk_id": f"chunk_{chunk_id:03}",
            "speaker": speaker_line,
            "text": speaker_text,
            "question": pending_question if pending_question else ""
        }

        chunks.append(chunk)
        chunk_id += 1
        pending_question = None  # Clear after attaching to next answer

    return chunks


def web_scrape(links):
    """
    Scrape the content of a web page given its URL.
    Input: links (str): The URLs of the web page to scrape.
    Output: combined chunks (str): The combined text content of the all web pages.
    """
    
    transcripts = []
    for link in links:
        transcripts.append(fetch_and_save_transcript(link))
    
    chunks = []
    for transcript in transcripts:
        chunks.extend(chunk_transcript(transcript))
    
    # Renumber the ids to be sequential across all transcripts
    for i in range(len(chunks)):
        chunks[i]["chunk_id"] = f"chunk_{i+1:03}"

    return chunks

In [57]:
links = [
    'https://www.fool.com/earnings/call-transcripts/2023/02/02/apple-aapl-q1-2023-earnings-call-transcript/',
    'https://www.fool.com/earnings/call-transcripts/2022/10/27/apple-aapl-q4-2022-earnings-call-transcript/',
    'https://www.fool.com/earnings/call-transcripts/2022/04/29/apple-aapl-q2-2022-earnings-call-transcript/',
    # 'https://www.fool.com/earnings/call-transcripts/2021/10/29/apple-aapl-q4-2021-earnings-call-transcript/',
    # 'https://www.fool.com/earnings/call-transcripts/2021/01/28/apple-aapl-q1-2021-earnings-call-transcript/',
    # 'https://www.fool.com/earnings/call-transcripts/2020/10/30/apple-aapl-q4-2020-earnings-call-transcript/',
    # 'https://www.fool.com/earnings/call-transcripts/2020/01/28/apple-inc-aapl-q1-2020-earnings-call-transcript.aspx'
]

chunks = web_scrape(links)

In [58]:
chunks

[{'chunk_id': 'chunk_001',
  'speaker': 'Tejas Gala -- Director, Investor Relations and Corporate Finance',
  'text': "Thank you. Speaking first today is Apple's CEO, Tim Cook; and he'll be followed by CFO, Luca Maestri. After that, we'll open the call to questions from analysts. Before turning the call over to Tim, I would like to remind everyone that the December quarter spanned 14 weeks, while the March quarter, as usual, has 13 weeks. Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, including, without limitation, those regarding revenue, gross margin, operating expenses, other income and expense, taxes, capital allocation, and future business outlook, including the potential impact of COVID-19 on the company's business and results of operations. These statements involve risks and uncertainties and that may cause actual results or trends to differ materially from our forecast. For more information, please re

## Create Query-Positive train set

In [29]:
import json
# Save to JSON
with open("train_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2)

Read in json file:

In [30]:
import json
json_path = "train_chunks.json"

# Open and load the JSON file
with open(json_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

In [59]:
import nltk

# Ensure we already filtered out short chunks
filtered_chunks = [chunk for chunk in chunks if len(nltk.word_tokenize(chunk["text"])) > 20]

In [33]:
len(filtered_chunks)

155

In [34]:
def generate_question_with_gpt4(answer_text: str) -> str:
    """
    Calls GPT-4-Turbo to generate one question for the given answer text.
    """
    # Craft a short system + user prompt
    messages = [
        {"role": "system", "content": "You are an expert question writer."},
        {
            "role": "user",
            "content": (
                "Given the following statement from an earnings call, "
                "write a concise question that would prompt this answer:\n\n"
                f"\"{answer_text}\"\n\n"
                "Only output the question."
            )
        }
    ]

    response = openai.ChatCompletion.create(
        model="gpt-4-turbo",
        messages=messages,
        temperature=0.7,
        max_tokens=64
    )
    # Extract the generated question
    question = response.choices[0].message.content.strip()
    return question



query_positive_pairs = []
for chunk in filtered_chunks:
    answer = chunk["text"]
    chunk_id = chunk["chunk_id"]
    question = generate_question_with_gpt4(answer)
    query_positive_pairs.append((question, chunk_id))

# Inspect a few examples
for q, cid in query_positive_pairs[:5]:
    print(f"Chunk ID: {cid}\nGenerated Question: {q}\n")

Chunk ID: chunk_001
Generated Question: Who will be speaking today during Apple's earnings call, and what important quarterly difference should listeners keep in mind?

Chunk ID: chunk_002
Generated Question: Could you provide an overview of Apple's financial performance and key highlights from the December quarter earnings report?

Chunk ID: chunk_003
Generated Question: What were the primary factors that contributed to the 5% decline in revenue for the December quarter, and how did these factors specifically impact the performance of key product categories like the iPhone 14 Pro?

Chunk ID: chunk_005
Generated Question: Can you describe the recent challenges in the supply chain, how the company has managed these issues, and what the outlook is for supply in the current quarter?

Chunk ID: chunk_007
Generated Question: Can you discuss how your margins performed in the December quarter and what your expectations are for margin trends moving into the March quarter?



In [54]:
len(query_positive_pairs)

155

In [36]:
query_positive_pairs

[("Who will be speaking today during Apple's earnings call, and what important quarterly difference should listeners keep in mind?",
  'chunk_001'),
 ("Could you provide an overview of Apple's financial performance and key highlights from the December quarter earnings report?",
  'chunk_002'),
 ('What were the primary factors that contributed to the 5% decline in revenue for the December quarter, and how did these factors specifically impact the performance of key product categories like the iPhone 14 Pro?',
  'chunk_003'),
 ('Can you describe the recent challenges in the supply chain, how the company has managed these issues, and what the outlook is for supply in the current quarter?',
  'chunk_005'),
 ('Can you discuss how your margins performed in the December quarter and what your expectations are for margin trends moving into the March quarter?',
  'chunk_007'),
 ('Can you clarify the impact of currency fluctuations and external factors like supply constraints and COVID restrictio

In [37]:
import json
# Save to JSON
with open("query_positive.json", "w", encoding="utf-8") as f:
    json.dump(query_positive_pairs, f, indent=2)

In [38]:
# Read from JSON
import json
json_path = "query_positive.json"

# Open and load the JSON file
with open(json_path, "r", encoding="utf-8") as f:
    query_positive_pairs = json.load(f)

## Fine-tuning

In [ ]:
# !pip install sentence-transformers rank_bm25 nltk

In [40]:
import random
import nltk
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from sentence_transformers.cross_encoder import CrossEncoder

In [41]:
# Tokenize corpus for BM25
nltk.download('punkt')
corpus_texts = [chunk['text'] for chunk in chunks]
tokenized_corpus = [nltk.word_tokenize(text.lower()) for text in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

# Cross‑encoder for hard negatives
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

[nltk_data] Downloading package punkt to C:\Users\Goh Ming
[nltk_data]     Wee\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [50]:
def sample_random_negative(pos_id, all_ids):
    """Random negative ≠ positive."""
    neg = random.choice([cid for cid in all_ids if cid != pos_id])
    return neg

def sample_bm25_negatives(query, pos_id, k=3):
    """Top‑k BM25 hits excluding the positive."""
    tokenized_query = nltk.word_tokenize(query.lower())
    scores = bm25.get_scores(tokenized_query)
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    negs = [chunks[i]['chunk_id'] for i in ranked if chunks[i]['chunk_id'] != pos_id][:k]
    return negs

def sample_cross_encoder_negatives(query, pos_id, candidate_ids, k=3):
    """Use cross‑encoder to pick top scoring negatives."""
    candidates = [cid for cid in candidate_ids if cid != pos_id]
    pairs = [[query, chunks_dict[cid]['text']] for cid in candidates]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in ranked[:k]]

In [51]:
len(filtered_chunks)

155

In [52]:
all_ids = [entry['chunk_id'] for entry in filtered_chunks]
chunks_dict = {entry['chunk_id']: entry for entry in filtered_chunks}
train_examples = []

for query, pos_id in query_positive_pairs:
    pos_text = chunks_dict[pos_id]['text']

    # 1) Random negative
    rand_id = sample_random_negative(pos_id, all_ids)
    train_examples.append(InputExample(texts=[query, pos_text, chunks_dict[rand_id]['text']]))

    # 2) BM25 hard negatives
    for neg_id in sample_bm25_negatives(query, pos_id, k=2):
        train_examples.append(InputExample(texts=[query, pos_text, chunks_dict[neg_id]['text']]))

    # 3) Cross-encoder hard negatives
    for neg_id in sample_cross_encoder_negatives(query, pos_id, all_ids, k=2):
        train_examples.append(InputExample(texts=[query, pos_text, chunks_dict[neg_id]['text']]))

In [53]:
from torch.utils.data import Dataset, DataLoader

# Load base bi‑encoder
model = SentenceTransformer("all-MiniLM-L6-v2")

# DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# # Choose a loss: TripletLoss for explicit triplets
# train_loss = losses.TripletLoss(model=model, triplet_margin=0.2)

# Alternatively, use in-batch negatives:
train_loss = losses.MultipleNegativesRankingLoss(model=model)

# Train
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=10
)

# Save
model.save("fine_tuned_earnings_retriever")

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


## Validation Evaluation

In [60]:
len(filtered_chunks)

69

In [62]:
import json
from langchain.schema import Document
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings


# Convert documents
documents = [
    Document(
        page_content=entry["text"],
        metadata={
            "chunk_id": entry["chunk_id"],
            "speaker": entry["speaker"],
            "question": entry["question"]
        }
    )
    for entry in chunks if entry["text"].strip()
]

# Initialize embedding model
# embedding_model = OpenAIEmbeddings(model="text-embedding-ada-002")
hf_embeddings = HuggingFaceEmbeddings(
    model_name="best_fine_tuned_earnings_retriever",
    model_kwargs={"device": "cpu"},             # e.g. "cuda" or "mps"
    encode_kwargs={"normalize_embeddings": True} # whether to L2‑normalize outputs
)


# Build FAISS index
# faiss_index = FAISS.from_documents(documents, embedding_model)
faiss_index = FAISS.from_documents(documents, hf_embeddings)

C:\Users\Goh Ming Wee\AppData\Local\Temp\ipykernel_14812\4013050905.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embeddings = HuggingFaceEmbeddings(


In [63]:
from sentence_transformers import CrossEncoder

# Load the cross-encoder model
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

In [64]:
def retrieve_relevant_docs(query, vector_store, k=5):
    # Retrieve top-k similar document chunks for the query
    # Semantic simiarity search using the vector store
    # This will return the top-k most relevant documents based on the query
    retrieved_docs = vector_store.similarity_search(query, k=k)
    return retrieved_docs


def rerank_documents(query, retrieved_docs, cross_encoder):
    # Prepare the inputs for the cross-encoder
    cross_encoder_inputs = [[query, doc.page_content] for doc in retrieved_docs]

    # Compute relevance scores
    relevance_scores = cross_encoder.predict(cross_encoder_inputs)

    # Attach scores to documents
    pairs_list = []
    for idx, doc in enumerate(retrieved_docs):
        pairs_list.append((doc, relevance_scores[idx]))

    # Sort documents by relevance score in descending order
    sorted_docs = sorted(pairs_list, key=lambda x: x[1], reverse=True)

    # Final output
    reranked_docs = [doc for doc, _ in sorted_docs]

    return reranked_docs

In [65]:
from rank_bm25 import BM25Okapi
import nltk
import numpy as np

# Download tokenizer data
nltk.download("punkt")

# Extract the raw text corpus
corpus = [doc.page_content for doc in documents]

# Tokenize each document (lowercased)
tokenized_corpus = [nltk.word_tokenize(text.lower()) for text in corpus]

# Instantiate BM25
bm25 = BM25Okapi(tokenized_corpus)

[nltk_data] Downloading package punkt to C:\Users\Goh Ming
[nltk_data]     Wee\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [66]:
def retrieve_bm25(query: str, k: int = 5):
    # Tokenize the query
    tokenized_query = nltk.word_tokenize(query.lower())
    # Get BM25 scores for every document
    scores = bm25.get_scores(tokenized_query)
    # Pick top‑k indices
    top_indices = np.argsort(scores)[::-1][:k]
    # Return the corresponding Document objects
    return [documents[i] for i in top_indices]

In [67]:
def hybrid_retrieve(query, k1=10, k2=10, final_k=5, vector_store=None, cross_encoder=None):      
    # BM25 retrieval
    bm25_docs = retrieve_bm25(query, k=k1)

    # Embedding retrieval
    emb_docs = retrieve_relevant_docs(query, faiss_index, k=k2)

    # Merge preserving order, remove duplicates by chunk_id
    seen = set()
    merged = []
    for doc in bm25_docs + emb_docs:
        cid = doc.metadata["chunk_id"]
        if cid not in seen:
            seen.add(cid)
            merged.append(doc)

    # Cross‑encode & rerank the merged set
    reranked = rerank_documents(query, merged, cross_encoder)

    # Return top‑final_k
    return reranked[:final_k]

In [68]:
from fuzzywuzzy import fuzz

test_queries = {
    "Q1": "When did Apple announce its fourth-quarter 2024 financial results?",
    "Q2": "What was Apple's total revenue for Q4 2024, and how did it compare year-over-year?",
    "Q3": "Which product categories set revenue records in Q4 2024?",
    "Q4": "What was the performance of Apple's Services segment in Q4 2024?",
    "Q5": "What is Apple Intelligence, and how is it being rolled out?",
    "Q6": "How did iPhone revenue perform in Q4 2024?",
    "Q7": "What were the key highlights of Apple's Wearables, Home, and Accessories segment?",
    "Q8": "What dividend did Apple declare for Q4 2024?",
    "Q9": "What was Apple's outlook for the December quarter (Q1 2025)?",
    "Q10": "What were the key health features announced for Apple Watch and AirPods?"
}

ground_truths = {
    "Q1": "Apple announced its Q4 2024 financial results on October 31, 2024.",
    "Q2": "Apple reported Q4 2024 revenue of $94.9 billion, up 6% year-over-year.",
    "Q3": "iPhone and Services set all-time revenue records for the September quarter, with segment records in the Americas, Europe, and rest of Asia Pacific.",
    "Q4": "Services revenue reached an all-time record of $25 billion, up 12% year-over-year, with growth across most categories.",
    "Q5": "Apple Intelligence is a personal AI system combining generative models with personal context. It is being rolled out in phases, starting with U.S. English in October 2024, expanding to more languages and features in December 2024 and beyond.",
    "Q6": "iPhone revenue was $46.2 billion, up 6% year-over-year, setting a September quarter record.",
    "Q7": "Apple Watch Series 10 introduced sleep apnea notifications, and AirPods Pro 2 added hearing health features like hearing tests and hearing aid capabilities.",
    "Q8": "Apple declared a cash dividend of $0.25 per share, payable on November 14, 2024.",
    "Q9": "Apple expects Q1 2025 revenue to grow low to mid-single digits year-over-year, with Services growing at a similar rate to fiscal 2024.",
    "Q10": "Apple Watch added sleep apnea notifications, while AirPods Pro 2 introduced hearing test and hearing aid features, described as 'life-changing' by users."
}

# Number of top results to retrieve
TOP_K = 5


# --- Evaluation Metrics ---

def fuzzy_match(text1, text2, threshold=50):
    """Returns True if two texts have high similarity (fuzzy match)."""
    return fuzz.partial_ratio(text1.lower(), text2.lower()) > threshold

def recall_at_k(retrieved_docs, relevant_doc, k):
    """Checks if relevant_doc is in top-K retrieved docs using fuzzy matching."""
    return int(any(fuzzy_match(relevant_doc, doc) for doc in retrieved_docs[:k]))

def precision_at_k(retrieved_docs, relevant_doc, k):
    """Fraction of retrieved docs that contain the relevant passage (fuzzy match)."""
    relevant_count = sum(1 for doc in retrieved_docs[:k] if fuzzy_match(relevant_doc, doc))
    return relevant_count / k if k > 0 else 0

def reciprocal_rank(retrieved_docs, relevant_doc):
    """Finds the first occurrence of relevant_doc using fuzzy matching."""
    for rank, doc in enumerate(retrieved_docs, start=1):
        if fuzzy_match(relevant_doc, doc):
            return 1 / rank
    return 0

c:\Users\Goh Ming Wee\AppData\Local\Programs\Python\Python39\lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [69]:
import numpy as np

# Store retrieved results
retrieved_results = {}

# Query the retriever
for q_id, query in test_queries.items():
    retrieved_docs = hybrid_retrieve(query, k1=5, k2=5, final_k=5, vector_store=faiss_index, cross_encoder=cross_encoder)
    retrieved_texts = [doc.page_content for doc in retrieved_docs]
    retrieved_results[q_id] = retrieved_texts

# --- Compute Metrics ---
recall_scores = []
precision_scores = []
mrr_scores = []

for q_id in test_queries.keys():
    retrieved_docs = retrieved_results[q_id]
    relevant_doc = ground_truths[q_id]

    recall_scores.append(recall_at_k(retrieved_docs, relevant_doc, TOP_K))
    precision_scores.append(precision_at_k(retrieved_docs, relevant_doc, TOP_K))
    mrr_scores.append(reciprocal_rank(retrieved_docs, relevant_doc))

# Compute mean scores
mean_recall_at_k = np.mean(recall_scores)
mean_precision_at_k = np.mean(precision_scores)
mrr = np.mean(mrr_scores)

# Print Results
print(f"Evaluation Metrics for RAG Retriever:")
print(f"----------------------------------")
print(f"Recall@{TOP_K}: {mean_recall_at_k:.2f}")
print(f"Precision@{TOP_K}: {mean_precision_at_k:.2f}")
print(f"MRR (Mean Reciprocal Rank): {mrr:.2f}")

Evaluation Metrics for RAG Retriever:
----------------------------------
Recall@5: 0.50
Precision@5: 0.10
MRR (Mean Reciprocal Rank): 0.14
